In [0]:
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("DataEntryLinage")

run_start = datetime.now()
logger.info(f"Data Entry SQL Linage pipeline started at {run_start}")

# Track step results for final summary
step_results = {}

In [0]:
# %sql
try:
    from pyspark.sql.functions import col, upper, trim, split, explode, size, element_at, regexp_extract_all, lit, when

# Step 1: Extract BO_TABLE and BO_SCHEMA from sql_definition
# sql_definition contains patterns like:
#   "TBBU_OVERALL_OUTPUT_IBR.FINAL_RATING"          → TABLE.COLUMN (2-part)
#   "APPS.AP_INVOICES_ALL.INVOICE_NUM"              → SCHEMA.TABLE.COLUMN (3-part)
#   "sum(TBPA_AGREEMENT_RESULTS.BC_CEDED_AMT)"     → TABLE.COLUMN inside function
#   "SUBSTR(APPS.AP_INVOICES_ALL.INVOICE_NUM,1,8)" → SCHEMA.TABLE.COLUMN inside function

df_entries = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_data_entries")

# Extract all dotted identifier chains (2+ parts: TABLE.COLUMN or SCHEMA.TABLE.COLUMN)
pattern = r"([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)+)"

df_extracted = (
    df_entries
    .filter(col("sql_definition").isNotNull() & (col("sql_definition") != "") & col("sql_definition").contains("."))
    .withColumn("dotted_chain", explode(regexp_extract_all(col("sql_definition"), lit(pattern))))
)

# Split chain and derive BO_TABLE / BO_SCHEMA:
#   Last part is always COLUMN (discard)
#   2-part (TABLE.COLUMN): BO_TABLE = part[0], BO_SCHEMA = NULL
#   3-part (SCHEMA.TABLE.COLUMN): BO_TABLE = part[1], BO_SCHEMA = part[0]
split_chain = split(col("dotted_chain"), r"\.")

df_step1 = (
    df_extracted
    .withColumn("_parts", split_chain)
    .withColumn("_size", size(col("_parts")))
    .filter(col("_size") >= 2)  # need at least TABLE.COLUMN
    .withColumn(
        "BO_TABLE",
        upper(trim(
            when(col("_size") == 2, element_at(col("_parts"), 1))   # TABLE.COLUMN → TABLE is 1st
            .when(col("_size") >= 3, element_at(col("_parts"), 2))  # SCHEMA.TABLE.COLUMN → TABLE is 2nd
        ))
    )
    .withColumn(
        "BO_SCHEMA",
        upper(trim(
            when(col("_size") >= 3, element_at(col("_parts"), 1))   # SCHEMA.TABLE.COLUMN → SCHEMA is 1st
        ))
    )
    .select("Document_Id", "Document_name", "BO_TABLE", "BO_SCHEMA")
    .distinct()
)

    row_count = df_step1.count()
    doc_count = df_step1.select('Document_Id').distinct().count()
    step_results['step1'] = {'rows': row_count, 'docs': doc_count, 'status': 'SUCCESS'}
    logger.info(f"Step 1 complete: {row_count} rows, {doc_count} unique documents")
    display(df_step1)
except Exception as e:
    step_results['step1'] = {'status': 'FAILED', 'error': str(e)}
    logger.error(f"Step 1 FAILED: {e}")
    raise

In [0]:
try:
    from pyspark.sql.functions import coalesce
    logger.info("Step 2: Joining cluster and Table_linage...")

    # Step 2: Join df_step1 with cluster details and Table_linage

# Get cluster for each Document_Id
df_clusters = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details")

# Get Table_linage for MI mapping
df_tl = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.Table_linage")

# Join cluster on Document_Id
df_with_cluster = (
    df_step1
    .join(
        df_clusters.select(
            upper(trim(col("Document_Id"))).alias("_cluster_doc_id"),
            col("cluster")
        ),
        upper(trim(df_step1["Document_Id"])) == col("_cluster_doc_id"),
        "left"
    )
    .drop("_cluster_doc_id")
)

# Join Table_linage on BO_TABLE = BO_SQL_TABLE
df_step2 = (
    df_with_cluster
    .join(
        df_tl.select(
            upper(trim(col("BO_SQL_TABLE"))).alias("_tl_bo_table"),
            upper(trim(col("MI_Table"))).alias("MI_Table"),
            upper(trim(col("MI_SCHEMA"))).alias("MI_SCHEMA"),
            upper(trim(col("MI_SOURCE"))).alias("MI_SOURCE"),
            upper(trim(col("Src_table"))).alias("Src_table"),
            upper(trim(col("Src_schema"))).alias("Src_schema"),
        ),
        upper(trim(df_with_cluster["BO_TABLE"])) == col("_tl_bo_table"),
        "left"
    )
    .drop("_tl_bo_table")
    .withColumn(
        "Calc_source_table",
        coalesce(col("Src_table"), col("MI_Table"), col("BO_TABLE"))
    )
    .withColumn(
        "Calc_source_schema",
        when(col("Src_table").isNull() & col("MI_Table").isNull(), col("BO_SCHEMA"))
        .when(col("Src_table").isNull(), col("MI_SCHEMA"))
        .otherwise(col("Src_schema"))
    )
    .select(
        "Document_Id", "Document_name", "cluster",
        "BO_TABLE", "BO_SCHEMA",
        "MI_Table", "MI_SCHEMA", "MI_SOURCE",
        "Src_table", "Src_schema",
        "Calc_source_table", "Calc_source_schema"
    )
    .distinct()
)

    row_count = df_step2.count()
    doc_count = df_step2.select('Document_Id').distinct().count()
    step_results['step2'] = {'rows': row_count, 'docs': doc_count, 'status': 'SUCCESS'}
    logger.info(f"Step 2 complete: {row_count} rows, {doc_count} unique documents")
    display(df_step2)
except Exception as e:
    step_results['step2'] = {'status': 'FAILED', 'error': str(e)}
    logger.error(f"Step 2 FAILED: {e}")
    raise

In [0]:
try:
    from pyspark.sql.functions import concat_ws, array_sort, collect_set
    logger.info("Step 3: Adding BO_DataConnection...")

    # Step 3: Add BO_DataConnection (aggregated Connection_Name per Document_Id)
df_cms = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")

df_connections = (
    df_cms
    .groupBy(upper(trim(col("Document_Id"))).alias("_conn_doc_id"))
    .agg(
        concat_ws("|", array_sort(collect_set(upper(trim(col("Connection_Name")))))).alias("BO_DataConnection")
    )
)

# Join to df_step2
df_step3 = (
    df_step2
    .join(
        df_connections,
        upper(trim(df_step2["Document_Id"])) == col("_conn_doc_id"),
        "left"
    )
    .drop("_conn_doc_id")
)

    row_count = df_step3.count()
    doc_count = df_step3.select('Document_Id').distinct().count()
    step_results['step3'] = {'rows': row_count, 'docs': doc_count, 'status': 'SUCCESS'}
    logger.info(f"Step 3 complete: {row_count} rows, {doc_count} unique documents")
    display(df_step3)
except Exception as e:
    step_results['step3'] = {'status': 'FAILED', 'error': str(e)}
    logger.error(f"Step 3 FAILED: {e}")
    raise

In [0]:
try:
    from pyspark.sql.functions import split as spark_split
    logger.info("Step 4: UC Schema mapping and ingestion flag...")

    # Step 4: Add Databricks_Schema + databricks_ingested flag, then write output

# UC Schema mapping
df_db_schema = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.applications_databrickUC_Schema_mapping")

# PRD information_schema for ingestion check
df_prd_tables = (
    spark.table("dataplatform01_central_prd_catalog_01.information_schema.tables")
    .select(
        upper(trim(col("table_schema"))).alias("_prd_schema"),
        upper(trim(col("table_name"))).alias("_prd_table")
    )
    .distinct()
)

# Join UC Schema mapping with same logic as Tables linage cell 7
df_with_schema = (
    df_step3
    .join(
        df_db_schema.select(
            upper(trim(col("BO_SOURCE"))).alias("_db_bo_source"),
            upper(trim(col("BO_source_schema"))).alias("_db_bo_schema"),
            col("Databricks_Schema")
        ),
        (
            (upper(trim(df_step3["Calc_source_schema"])) == col("_db_bo_schema"))
            & (
                (upper(trim(df_step3["Calc_source_schema"])) != "ORABUP0")
                | (
                    (upper(trim(df_step3["Calc_source_schema"])) == "ORABUP0")
                    & (upper(trim(spark_split(df_step3["MI_SOURCE"], " ")[0])) == col("_db_bo_source"))
                )
            )
        ),
        "left"
    )
    .drop("_db_bo_source", "_db_bo_schema")
)

# Join PRD tables to check if table is ingested
df_step4 = (
    df_with_schema
    .join(
        df_prd_tables,
        (upper(trim(col("Databricks_Schema"))) == col("_prd_schema"))
        & (upper(trim(df_with_schema["Calc_source_table"])) == col("_prd_table")),
        "left"
    )
    .withColumn(
        "databricks_ingested",
        when(col("_prd_schema").isNull() | col("_prd_table").isNull(), "N").otherwise("Y")
    )
    .drop("_prd_schema", "_prd_table")
    .distinct()
)

    row_count = df_step4.count()
    doc_count = df_step4.select('Document_Id').distinct().count()
    logger.info(f"Step 4 result: {row_count} rows, {doc_count} unique documents")

    # Write to target table
    df_step4.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        "dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_dataentry_linage_UCflagged"
    )
    step_results['step4'] = {'rows': row_count, 'docs': doc_count, 'status': 'SUCCESS'}
    logger.info("Step 4 complete: written to active_webi_dataentry_linage_UCflagged")
    display(df_step4)
except Exception as e:
    step_results['step4'] = {'status': 'FAILED', 'error': str(e)}
    logger.error(f"Step 4 FAILED: {e}")
    raise

In [0]:
%sql

describe table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_UCflagged

In [0]:
try:
    from pyspark.sql.functions import lit, row_number, desc
    from pyspark.sql.window import Window
    logger.info("Step 5: Combining both linage tables...")

    # Step 5: Combine active_webi_dataentry_linage_UCflagged + active_webi_full_linage_UCflagged
# Dedup: prioritize 'Data entry definition' over 'SQL extracted'

df_dataentry = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_dataentry_linage_UCflagged")
df_sql_extracted = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_UCflagged")

# Select common columns + source tag
common_cols = [
    "Document_Id", "Document_name", "cluster", 
    "BO_TABLE", "BO_SCHEMA",
    "Calc_source_table", "Calc_source_schema",
    "BO_DataConnection", "Databricks_Schema", "databricks_ingested"
]

df_de = (
    df_dataentry
    .select(*common_cols)
    .withColumn("SAP_BO_Connection", lit(None).cast("string"))
    .withColumn("source_type", lit("Data entry definition"))
)

df_sql = (
    df_sql_extracted
    .select(*common_cols, "SAP_BO_Connection")
    .withColumn("source_type", lit("SQL extracted"))
)

# Union both
final_cols = [
    "Document_Id", "Document_name", "cluster", "SAP_BO_Connection",
    "BO_TABLE", "BO_SCHEMA",
    "Calc_source_table", "Calc_source_schema",
    "BO_DataConnection", "Databricks_Schema", "databricks_ingested", "source_type"
]

df_combined = df_de.select(*final_cols).unionByName(df_sql.select(*final_cols))

# Dedup: on (Document_Id, BO_TABLE, Calc_source_table), prioritize 'Data entry definition'
w_dedup = Window.partitionBy("Document_Id", "BO_TABLE", "Calc_source_table").orderBy(
    when(col("source_type") == "Data entry definition", 0).otherwise(1)
)

df_final = (
    df_combined
    .withColumn("_rn", row_number().over(w_dedup))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .withColumnRenamed("Calc_source_table", "Final_table")
    .withColumnRenamed("Calc_source_schema", "Final_schema")
    .distinct()
)

    row_count = df_final.count()
    doc_count = df_final.select('Document_Id').distinct().count()
    step_results['step5'] = {'rows': row_count, 'docs': doc_count, 'status': 'SUCCESS'}
    logger.info(f"Step 5 complete: {row_count} rows, {doc_count} unique documents")
    df_final.groupBy("source_type").count().show()
    display(df_final)
except Exception as e:
    step_results['step5'] = {'status': 'FAILED', 'error': str(e)}
    logger.error(f"Step 5 FAILED: {e}")
    raise

In [0]:
try:
    from pyspark.sql.functions import row_number, current_timestamp
    from pyspark.sql.window import Window
    logger.info("Step 6: Arcade impact assessment...")

    # Step 6: Arcade impact assessment on the combined final output
# Same logic as Arcade_Impact_assessment notebook

df_release_scope = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_scope")
df_release_list = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.arcaderisk_release_list")

# --- 6a: Project impact per report (using release_scope) ---
base = (
    df_final
    .join(
        df_release_scope.select(
            upper(trim(col("mi_table"))).alias("_scope_table"),
            col("Impacted_Flag"),
            col("Release_scheduled")
        ),
        col("Final_table") == col("_scope_table"),
        "left"
    )
    .drop("_scope_table")
    .withColumn(
        "Prioritized",
        when((upper(trim(col("Impacted_Flag"))) == "Y") & (upper(trim(col("Release_scheduled"))) == "Y"), "Y")
        .when((upper(trim(col("Impacted_Flag"))) == "Y") & (upper(trim(col("Release_scheduled"))) == "N"), "N")
        .otherwise(None)
    )
    .withColumn(
        "Impacted",
        when((upper(trim(col("Impacted_Flag"))) == "N") | col("Impacted_Flag").isNull(), "N").otherwise("Y")
    )
)

# Dedup: keep one row per Document_Id (Impacted=Y wins, then Prioritized=N wins over Y wins over null)
w_impact = Window.partitionBy("Document_Id").orderBy(
    desc("Impacted"),
    when(col("Prioritized") == "N", 1).when(col("Prioritized") == "Y", 2).otherwise(3)
)

df_impact_report = (
    base
    .withColumn("_rn", row_number().over(w_impact))
    .filter(col("_rn") == 1)
    .drop("_rn", "Impacted_Flag", "Release_scheduled")
    .withColumn("ingestion_ts", current_timestamp())
)

print(f"Impact report: {df_impact_report.count()} rows, {df_impact_report.select('Document_Id').distinct().count()} unique documents")
print("Impact breakdown:")
df_impact_report.groupBy("Impacted", "Prioritized").count().orderBy("Impacted", "Prioritized").show()

# --- 6b: Release-level impact per report (using release_list) ---
base_release = (
    df_final
    .join(
        df_release_list.select(
            upper(trim(col("mi_table"))).alias("_rel_table"),
            col("Impacted_Flag").alias("_rel_impact"),
            col("Release_schedule")
        ),
        col("Final_table") == col("_rel_table"),
        "left"
    )
    .drop("_rel_table")
    .withColumn(
        "Prioritized",
        when((upper(trim(col("_rel_impact"))) == "Y") & col("Release_schedule").isNotNull(), "Y")
        .when((upper(trim(col("_rel_impact"))) == "Y") & col("Release_schedule").isNull(), "N")
        .otherwise(None)
    )
    .withColumn(
        "Impacted",
        when((upper(trim(col("_rel_impact"))) == "N") | col("_rel_impact").isNull(), "N").otherwise("Y")
    )
    .drop("_rel_impact")
)

w_release = Window.partitionBy("Document_Id").orderBy(
    desc("Impacted"),
    when(col("Prioritized") == "N", 1).when(col("Prioritized") == "Y", 2).otherwise(3),
    desc("Release_schedule")
)

df_release_impact = (
    base_release
    .withColumn("_rn", row_number().over(w_release))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .withColumn("ingestion_ts", current_timestamp())
)

print(f"\nRelease impact: {df_release_impact.count()} rows")
print("Release breakdown:")
df_release_impact.groupBy("Impacted", "Release_schedule").count().orderBy("Impacted", "Release_schedule").show()

# --- Write outputs ---
df_impact_report.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "dataplatform01_central_dev_catalog_01.custom_sap_bo.combined_linage_project_impact_report"
)
    logger.info("Written: combined_linage_project_impact_report")

    df_release_impact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        "dataplatform01_central_dev_catalog_01.custom_sap_bo.combined_linage_release_impact_report"
    )
    logger.info("Written: combined_linage_release_impact_report")

    step_results['step6'] = {
        'impact_rows': df_impact_report.count(),
        'release_rows': df_release_impact.count(),
        'status': 'SUCCESS'
    }
    logger.info("Step 6 complete.")
except Exception as e:
    step_results['step6'] = {'status': 'FAILED', 'error': str(e)}
    logger.error(f"Step 6 FAILED: {e}")
    raise

In [0]:
from datetime import datetime

run_end = datetime.now()
duration = run_end - run_start

logger.info("="*60)
logger.info("PIPELINE SUMMARY")
logger.info("="*60)
logger.info(f"Total duration: {duration}")

failed_steps = [k for k, v in step_results.items() if v.get('status') == 'FAILED']

for step, result in step_results.items():
    status = result.get('status', 'UNKNOWN')
    rows = result.get('rows', result.get('impact_rows', '-'))
    docs = result.get('docs', '-')
    error = result.get('error', '')
    if status == 'SUCCESS':
        logger.info(f"  {step}: {status} | rows={rows}, docs={docs}")
    else:
        logger.error(f"  {step}: {status} | error={error}")

if failed_steps:
    logger.error(f"\nPIPELINE COMPLETED WITH ERRORS in: {', '.join(failed_steps)}")
else:
    logger.info(f"\nPIPELINE COMPLETED SUCCESSFULLY - all {len(step_results)} steps passed")